# 08 — Advanced OOP in Python

**Level:** Intermediate → Advanced

This notebook builds on the OOP fundamentals and introduces advanced Python OOP concepts used in real applications.

### Topics
- Multiple inheritance
- Method Resolution Order (MRO)
- `super()`
- Composition and aggregation
- Association
- Abstract base classes
- Interfaces and protocols
- Properties
- Class/static methods
- Dataclasses
- Custom exceptions
- Dependency injection
- Mixins
- SOLID principles
- Design patterns
- Advanced ML-oriented OOP


## 1. Multiple Inheritance

A class can inherit from more than one parent class.

```python
class Child(Parent1, Parent2):
    pass
```

Use multiple inheritance carefully because method resolution can become complex.


In [ ]:
class Father:
    def skills(self):
        return "Driving"

class Mother:
    def hobbies(self):
        return "Cooking"

class Child(Father, Mother):
    pass

child = Child()

print(child.skills())
print(child.hobbies())


## 2. Method Resolution Order (MRO)

MRO tells Python the order in which it searches classes for methods and attributes.

Use:

```python
ClassName.mro()
```

or:

```python
ClassName.__mro__
```


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")

class C(B):
    pass

print(C.mro())

obj = C()
obj.show()


## 3. Diamond Inheritance

The diamond problem occurs when multiple inheritance creates a shared ancestor.

Python resolves this using its MRO algorithm.


In [ ]:
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")
        super().show()

class C(A):
    def show(self):
        print("C")
        super().show()

class D(B, C):
    def show(self):
        print("D")
        super().show()

print(D.mro())

D().show()


## 4. `super()`

`super()` gives access to the next implementation according to the MRO.

It is especially useful when extending parent behavior rather than completely replacing it.


In [ ]:
class Employee:
    def __init__(self, name):
        self.name = name

class Developer(Employee):
    def __init__(self, name, language):
        super().__init__(name)
        self.language = language

developer = Developer("Ajim", "Python")

print(developer.name)
print(developer.language)


## 5. Composition

**Composition** means building a class using objects of other classes.

Instead of:

```text
Car IS-A Engine
```

we model:

```text
Car HAS-A Engine
```

Composition is often more flexible than deep inheritance hierarchies.


In [ ]:
class Engine:
    def start(self):
        return "Engine started"

class Car:
    def __init__(self):
        self.engine = Engine()

    def start(self):
        return self.engine.start()

car = Car()
print(car.start())


## 6. Aggregation

Aggregation is also a HAS-A relationship, but the contained object can exist independently.

Example:

```text
Department HAS-A Teacher
```

A teacher can exist even if the department object is removed.


In [ ]:
class Teacher:
    def __init__(self, name):
        self.name = name

class Department:
    def __init__(self, name, teachers):
        self.name = name
        self.teachers = teachers

teachers = [Teacher("A"), Teacher("B")]
department = Department("Data Science", teachers)

print(department.name)
print([teacher.name for teacher in department.teachers])


## 7. Association

Association means two objects interact with each other without one necessarily owning the other.

Example:

```text
Doctor ↔ Patient
Teacher ↔ Student
Customer ↔ Bank
```


In [ ]:
class Student:
    def __init__(self, name):
        self.name = name

class Teacher:
    def teach(self, student):
        print(f"{self.__class__.__name__} teaches {student.name}")

student = Student("Ajim")
teacher = Teacher()

teacher.teach(student)


## 8. Composition vs Inheritance

### Inheritance
Use when the relationship is genuinely **IS-A**.

```text
Dog IS-A Animal
```

### Composition
Use when the relationship is **HAS-A**.

```text
Car HAS-A Engine
```

A useful design rule is:

> Prefer composition when it gives simpler and more flexible designs.


## 9. Properties

Properties let you expose methods using attribute-like syntax.

They are useful for validation, computed values, and controlled access.


In [ ]:
class Employee:
    def __init__(self, salary):
        self._salary = salary

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, value):
        if value < 0:
            raise ValueError("Salary cannot be negative")
        self._salary = value

employee = Employee(50000)

print(employee.salary)

employee.salary = 60000
print(employee.salary)


## 10. Read-Only Property

A property without a setter behaves like a read-only attribute from the class API.


In [ ]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14159 * self.radius ** 2

circle = Circle(5)

print(circle.area)


## 11. Class Methods

A class method receives `cls` and operates at the class level.

Use:

```python
@classmethod
```

A common use is an alternative constructor.


In [ ]:
class Employee:
    company = "TechCorp"

    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    @classmethod
    def from_string(cls, text):
        name, salary = text.split(",")
        return cls(name, int(salary))

employee = Employee.from_string("Ajim,60000")

print(employee.name)
print(employee.salary)
print(employee.company)


## 12. Static Methods

A static method does not receive `self` or `cls` automatically.

Use:

```python
@staticmethod
```

It is useful when a function logically belongs to a class but does not need object or class state.


In [ ]:
class MathUtils:
    @staticmethod
    def is_even(number):
        return number % 2 == 0

print(MathUtils.is_even(10))
print(MathUtils.is_even(7))


## 13. Instance vs Class vs Static Methods

| Type | First Parameter | Access |
|---|---|---|
| Instance method | `self` | Instance state |
| Class method | `cls` | Class state |
| Static method | None automatically | Independent utility logic |


## 14. Abstract Base Classes

Abstract classes define a contract for subclasses.

Python provides:

```python
from abc import ABC, abstractmethod
```


In [ ]:
from abc import ABC, abstractmethod

class Payment(ABC):
    @abstractmethod
    def pay(self, amount):
        pass

class UPI(Payment):
    def pay(self, amount):
        return f"Paid ₹{amount} using UPI"

payment = UPI()
print(payment.pay(1000))


## 15. Dataclasses

`dataclasses` reduce boilerplate for classes that primarily store data.

They can automatically provide methods such as:

- `__init__()`
- `__repr__()`
- `__eq__()`

depending on configuration.


In [ ]:
from dataclasses import dataclass

@dataclass
class Student:
    name: str
    marks: float

student = Student("Ajim", 92)

print(student)
print(student.name)
print(student.marks)


## 16. Dataclass with Validation Logic

Dataclasses do not replace all normal classes. You can still add methods and validation.


In [ ]:
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

    def __post_init__(self):
        if self.price < 0:
            raise ValueError("Price cannot be negative")

product = Product("Laptop", 75000)
print(product)


## 17. Custom Exceptions in OOP

Custom exceptions make domain-specific errors clearer.


In [ ]:
class InsufficientBalanceError(Exception):
    pass

class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientBalanceError("Insufficient balance")
        self.balance -= amount

account = BankAccount(1000)

try:
    account.withdraw(1500)
except InsufficientBalanceError as error:
    print(error)


## 18. Dependency Injection

Dependency injection means a class receives the object it depends on instead of creating that dependency internally.

This improves:

- testing
- flexibility
- maintainability
- loose coupling


In [ ]:
class EmailService:
    def send(self, message):
        print("Email:", message)

class NotificationManager:
    def __init__(self, service):
        self.service = service

    def notify(self, message):
        self.service.send(message)

service = EmailService()
manager = NotificationManager(service)

manager.notify("Model training completed")


## 19. Why Dependency Injection Helps Testing

We can replace the real dependency with a fake implementation.


In [ ]:
class FakeService:
    def __init__(self):
        self.messages = []

    def send(self, message):
        self.messages.append(message)

fake = FakeService()
manager = NotificationManager(fake)

manager.notify("Test message")

print(fake.messages)


## 20. Mixins

A mixin is a small class designed to provide reusable behavior to another class.

A mixin normally represents a capability rather than a complete business entity.


In [ ]:
class LoggingMixin:
    def log(self, message):
        print(f"[LOG] {message}")

class ModelTrainer(LoggingMixin):
    def train(self):
        self.log("Training started")
        self.log("Training completed")

trainer = ModelTrainer()
trainer.train()


## 21. SOLID Principles

SOLID is a group of object-oriented design principles.

### S — Single Responsibility Principle
A class should have one primary responsibility.

### O — Open/Closed Principle
Software should be open for extension but closed for unnecessary modification.

### L — Liskov Substitution Principle
Subtypes should be usable wherever their base types are expected.

### I — Interface Segregation Principle
Prefer focused interfaces instead of large interfaces clients do not need.

### D — Dependency Inversion Principle
Depend on abstractions rather than concrete implementations.


## 22. Single Responsibility Example

Avoid one huge class that handles:

```text
data loading
cleaning
training
evaluation
reporting
email
database storage
```

Split responsibilities into focused components.


In [ ]:
class DataLoader:
    def load(self):
        return [[1], [2], [3]]

class Model:
    def train(self, data):
        print("Training on", data)

class Evaluator:
    def evaluate(self, model, data):
        print("Evaluating model")
        return 0.90

data = DataLoader().load()
model = Model()
model.train(data)

score = Evaluator().evaluate(model, data)
print("Score:", score)


## 23. Strategy Pattern

The Strategy pattern lets you switch algorithms without changing the main class.

This is especially useful in Data Science when different algorithms perform the same conceptual task.


In [ ]:
class LogisticStrategy:
    def train(self, X, y):
        return "Logistic Regression trained"

class TreeStrategy:
    def train(self, X, y):
        return "Decision Tree trained"

class Trainer:
    def __init__(self, strategy):
        self.strategy = strategy

    def train(self, X, y):
        return self.strategy.train(X, y)

trainer = Trainer(LogisticStrategy())
print(trainer.train([[1], [2]], [0, 1]))

trainer.strategy = TreeStrategy()
print(trainer.train([[1], [2]], [0, 1]))


## 24. Factory Pattern

A factory centralizes object creation.

This avoids scattering class-selection logic throughout an application.


In [ ]:
class LogisticModel:
    def predict(self, X):
        return "Logistic prediction"

class TreeModel:
    def predict(self, X):
        return "Tree prediction"

class ModelFactory:
    @staticmethod
    def create(model_type):
        if model_type == "logistic":
            return LogisticModel()
        if model_type == "tree":
            return TreeModel()
        raise ValueError("Unknown model type")

model = ModelFactory.create("logistic")
print(model.predict([[1, 2]]))


## 25. Observer Pattern — Concept

The Observer pattern allows one object to notify multiple dependent objects when an event occurs.

Common examples:

- notifications
- event systems
- UI updates
- training callbacks
- monitoring systems


In [ ]:
class Observer:
    def update(self, message):
        raise NotImplementedError

class EmailObserver(Observer):
    def update(self, message):
        print("Email:", message)

class LoggerObserver(Observer):
    def update(self, message):
        print("Log:", message)

class TrainingProcess:
    def __init__(self):
        self.observers = []

    def subscribe(self, observer):
        self.observers.append(observer)

    def notify(self, message):
        for observer in self.observers:
            observer.update(message)

process = TrainingProcess()
process.subscribe(EmailObserver())
process.subscribe(LoggerObserver())

process.notify("Training finished")


## 26. Advanced ML OOP Architecture

A maintainable ML system can separate responsibilities:

```text
DataLoader
    ↓
Preprocessor
    ↓
FeatureEngineer
    ↓
Model
    ↓
Evaluator
    ↓
ModelSerializer
    ↓
PredictionService
```

Each component has a focused responsibility.

This is much easier to test and maintain than putting the entire ML workflow in one class.


In [ ]:
from abc import ABC, abstractmethod

class Preprocessor(ABC):
    @abstractmethod
    def transform(self, X):
        pass

class StandardScalerProcessor(Preprocessor):
    def transform(self, X):
        return X

class Model(ABC):
    @abstractmethod
    def train(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass

class DummyClassifier(Model):
    def train(self, X, y):
        self.classes = sorted(set(y))
        return self

    def predict(self, X):
        return [self.classes[0] for _ in X]

processor = StandardScalerProcessor()
model = DummyClassifier()

X = processor.transform([[1, 2], [3, 4]])
model.train(X, [0, 1])

print(model.predict([[5, 6]]))


## 27. Protocols — Structural Typing

Python's `typing.Protocol` can describe an expected interface without requiring inheritance.

This supports a concept called **structural typing**: an object is acceptable when it provides the required behavior.


In [ ]:
from typing import Protocol

class Predictor(Protocol):
    def predict(self, X):
        ...

class SimplePredictor:
    def predict(self, X):
        return [0 for _ in X]

def run_prediction(model: Predictor, X):
    return model.predict(X)

model = SimplePredictor()
print(run_prediction(model, [[1], [2]]))


## 28. `__slots__` Revisited

`__slots__` can restrict dynamically created instance attributes and may save memory for many instances.

Use it only when the trade-offs are understood.

It can affect features such as instance `__dict__` and weak references.


In [ ]:
class Coordinate:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

point = Coordinate(10, 20)
print(point.x, point.y)


## 29. Advanced Encapsulation

Python does not enforce private attributes in the same way some languages do.

Conventions include:

```text
name       → public
_name      → internal/non-public convention
__name     → name mangling
```

Name mangling is mainly intended to avoid accidental name collisions in subclasses; it is not a security mechanism.


In [ ]:
class Account:
    def __init__(self, balance):
        self.__balance = balance

    def get_balance(self):
        return self.__balance

account = Account(5000)

print(account.get_balance())
print(account._Account__balance)  # Demonstrates name mangling; avoid using this in normal code.


## 30. Advanced Challenge — Pluggable ML Models

Design a system where multiple model classes implement the same interface:

```python
train(X, y)
predict(X)
evaluate(X, y)
```

Add:

- abstract base class
- model metadata
- `__repr__()`
- `__call__()`
- dependency injection for preprocessing
- a factory for model creation


In [ ]:
from abc import ABC, abstractmethod

class Model(ABC):
    @abstractmethod
    def train(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    def evaluate(self, X, y):
        predictions = self.predict(X)
        return sum(p == actual for p, actual in zip(predictions, y)) / len(y)

    def __call__(self, X):
        return self.predict(X)

class MajorityModel(Model):
    def train(self, X, y):
        self.majority = max(set(y), key=y.count)
        return self

    def predict(self, X):
        if not hasattr(self, "majority"):
            raise RuntimeError("Train the model first")
        return [self.majority] * len(X)

    def __repr__(self):
        return "MajorityModel()"

model = MajorityModel()
model.train([[1], [2], [3]], [0, 0, 1])

print(model)
print(model([[4], [5]]))
print(model.evaluate([[4], [5]], [0, 0]))


## 31. Practice Tasks

### Task 1 — Bank System
Build:

- `Bank`
- `Account`
- custom exception
- deposit/withdraw methods
- property for balance

### Task 2 — E-Commerce
Build:

- `Product`
- `Cart`
- `Order`
- payment abstraction
- notification service

### Task 3 — ML System
Build:

- abstract `Model`
- two model implementations
- preprocessing component
- evaluator
- factory
- prediction service

Focus on **composition and clean responsibilities**, not just making the code run.


## 32. Interview Questions

1. What is multiple inheritance?
2. What is MRO?
3. How does `super()` work?
4. What is the diamond problem?
5. Composition vs inheritance?
6. Association vs aggregation?
7. What are class methods and static methods?
8. What is dependency injection?
9. What is a mixin?
10. What are SOLID principles?
11. What is a dataclass?
12. What is a protocol?
13. What is structural typing?
14. Why prefer composition in many designs?
15. How would you design an extensible ML model architecture?


## 33. Final Mental Model

```text
OOP Fundamentals
      ↓
Encapsulation
      ↓
Inheritance
      ↓
Polymorphism
      ↓
Abstraction
      ↓
Magic Methods
      ↓
Advanced Design
      ├── Composition
      ├── MRO / Multiple Inheritance
      ├── Properties
      ├── Dataclasses
      ├── Dependency Injection
      ├── Mixins
      ├── Protocols
      ├── SOLID
      └── Design Patterns
```

### Key Principle

> Good OOP is not about creating the maximum number of classes. It is about creating clear responsibilities and useful relationships between objects.


# 09 — OOP Real-World Project

## Project: Machine Learning Model Management System

This project combines the OOP concepts learned in notebooks 01–08.

### Goal

Build a small, extensible ML-style application using:

- Encapsulation
- Inheritance
- Polymorphism
- Abstraction
- Magic methods
- Composition
- Dependency injection
- Factory pattern
- Custom exceptions

The project intentionally uses lightweight Python logic so it can run without external ML packages.


## 1. Project Architecture

```text
ML Model Management System
│
├── DataManager
│
├── Preprocessor
│       └── StandardPreprocessor
│
├── Model (Abstract)
│       ├── ClassificationModel
│       └── RegressionModel
│
├── Evaluator
│
├── ModelFactory
│
└── MLApplication
```

The architecture demonstrates how OOP concepts combine in a practical system.


## 2. Custom Exceptions

Define application-specific errors first.


In [ ]:
class ModelNotTrainedError(Exception):
    pass

class UnknownModelError(Exception):
    pass

class InvalidDataError(Exception):
    pass


## 3. Data Manager

The `DataManager` handles dataset storage and validation.

This demonstrates **encapsulation** and `__len__()`.


In [ ]:
class DataManager:
    def __init__(self, X, y):
        self._X = list(X)
        self._y = list(y)

        if len(self._X) != len(self._y):
            raise InvalidDataError("X and y must have the same length")

    @property
    def X(self):
        return self._X

    @property
    def y(self):
        return self._y

    def __len__(self):
        return len(self._X)

    def __getitem__(self, index):
        return self._X[index], self._y[index]

    def __repr__(self):
        return f"DataManager(samples={len(self)})"

data = DataManager(
    [[1, 2], [2, 3], [3, 4], [4, 5]],
    [0, 0, 1, 1]
)

print(data)
print(len(data))
print(data[0])


## 4. Preprocessor Abstraction

A preprocessing component defines a common interface.


In [ ]:
from abc import ABC, abstractmethod

class Preprocessor(ABC):
    @abstractmethod
    def fit(self, X):
        pass

    @abstractmethod
    def transform(self, X):
        pass

    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)


## 5. Concrete Preprocessor

A simple scaler demonstrates inheritance and implementation of an abstract interface.


In [ ]:
class StandardPreprocessor(Preprocessor):
    def fit(self, X):
        if not X:
            raise InvalidDataError("X cannot be empty")

        columns = list(zip(*X))
        self.means = [sum(col) / len(col) for col in columns]

        self.stds = []
        for col, mean in zip(columns, self.means):
            variance = sum((value - mean) ** 2 for value in col) / len(col)
            std = variance ** 0.5
            self.stds.append(std if std != 0 else 1)

        return self

    def transform(self, X):
        if not hasattr(self, "means"):
            raise RuntimeError("Preprocessor must be fitted first")

        return [
            [(value - mean) / std for value, mean, std in zip(row, self.means, self.stds)]
            for row in X
        ]

preprocessor = StandardPreprocessor()
X_scaled = preprocessor.fit_transform(data.X)

print(X_scaled)


## 6. Model Abstraction

Every model must provide:

```python
train(X, y)
predict(X)
```

The base class also supplies evaluation and callable behavior.


In [ ]:
class Model(ABC):
    @abstractmethod
    def train(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    def evaluate(self, X, y):
        predictions = self.predict(X)
        if len(predictions) != len(y):
            raise InvalidDataError("Prediction and target lengths differ")

        return sum(p == actual for p, actual in zip(predictions, y)) / len(y)

    def __call__(self, X):
        return self.predict(X)


## 7. Classification Model

This simple classifier learns the majority class. It is intentionally lightweight; the focus is the OOP architecture.


In [ ]:
class ClassificationModel(Model):
    def __init__(self):
        self._trained = False

    def train(self, X, y):
        if not y:
            raise InvalidDataError("Training labels cannot be empty")

        self._majority_class = max(set(y), key=y.count)
        self._trained = True
        return self

    def predict(self, X):
        if not self._trained:
            raise ModelNotTrainedError("Train the classifier first")

        return [self._majority_class] * len(X)

    def __bool__(self):
        return self._trained

    def __repr__(self):
        return f"ClassificationModel(trained={self._trained})"


## 8. Regression Model

The regression model predicts the average target value.

Again, the algorithm is intentionally simple so that the OOP design remains the main focus.


In [ ]:
class RegressionModel(Model):
    def __init__(self):
        self._trained = False

    def train(self, X, y):
        if not y:
            raise InvalidDataError("Training targets cannot be empty")

        self._mean_target = sum(y) / len(y)
        self._trained = True
        return self

    def predict(self, X):
        if not self._trained:
            raise ModelNotTrainedError("Train the regression model first")

        return [self._mean_target] * len(X)

    def __bool__(self):
        return self._trained

    def __repr__(self):
        return f"RegressionModel(trained={self._trained})"


## 9. Polymorphism

Both models support:

```python
train()
predict()
evaluate()
```

The application can work with either model through the common `Model` interface.


In [ ]:
classification = ClassificationModel()
classification.train(X_scaled, data.y)

print(classification)
print(classification.predict(X_scaled))
print(classification.evaluate(X_scaled, data.y))

regression = RegressionModel()
regression.train(X_scaled, [10, 20, 30, 40])

print(regression)
print(regression.predict(X_scaled))
print(regression.evaluate(X_scaled, [10, 20, 30, 40]))


## 10. Model Factory

The factory creates models from a simple name.


In [ ]:
class ModelFactory:
    @staticmethod
    def create(model_type):
        if model_type == "classification":
            return ClassificationModel()

        if model_type == "regression":
            return RegressionModel()

        raise UnknownModelError(f"Unknown model type: {model_type}")

model1 = ModelFactory.create("classification")
model2 = ModelFactory.create("regression")

print(model1)
print(model2)


## 11. Evaluator

Keep evaluation as a separate responsibility.

This demonstrates the **Single Responsibility Principle**.


In [ ]:
class Evaluator:
    def evaluate(self, model, X, y):
        if not model:
            raise ModelNotTrainedError("Cannot evaluate an untrained model")

        return model.evaluate(X, y)

evaluator = Evaluator()

score = evaluator.evaluate(
    classification,
    X_scaled,
    data.y
)

print("Classification score:", score)


## 12. Application Service

Now use composition to combine:

- data manager
- preprocessor
- model
- evaluator

This is the core of the project.


In [ ]:
class MLApplication:
    def __init__(self, data_manager, preprocessor, model, evaluator):
        self.data_manager = data_manager
        self.preprocessor = preprocessor
        self.model = model
        self.evaluator = evaluator

    def train(self):
        X = self.preprocessor.fit_transform(self.data_manager.X)
        self.model.train(X, self.data_manager.y)
        return self

    def evaluate(self):
        X = self.preprocessor.transform(self.data_manager.X)
        return self.evaluator.evaluate(
            self.model,
            X,
            self.data_manager.y
        )

    def predict(self, X):
        X_transformed = self.preprocessor.transform(X)
        return self.model(X_transformed)

    def __repr__(self):
        return (
            f"MLApplication(data={self.data_manager}, "
            f"model={self.model})"
        )


## 13. Run the Complete Application

The application uses dependency injection: all components are supplied from outside.


In [ ]:
data = DataManager(
    [[1, 2], [2, 3], [3, 4], [4, 5]],
    [0, 0, 1, 1]
)

app = MLApplication(
    data_manager=data,
    preprocessor=StandardPreprocessor(),
    model=ModelFactory.create("classification"),
    evaluator=Evaluator()
)

print(app)

app.train()

print("Evaluation:", app.evaluate())
print("Predictions:", app.predict([[5, 6], [6, 7]]))


## 14. Switch the Model Without Rewriting the Application

Because the application depends on the abstract `Model` behavior, we can replace the concrete model.


In [ ]:
regression_data = DataManager(
    [[1], [2], [3], [4]],
    [10, 20, 30, 40]
)

regression_app = MLApplication(
    regression_data,
    StandardPreprocessor(),
    ModelFactory.create("regression"),
    Evaluator()
)

regression_app.train()

print(regression_app)
print("Regression predictions:", regression_app.predict([[5], [6]]))


## 15. Project Workflow

```text
Input Data
    ↓
DataManager
    ↓
Preprocessor
    ↓
ModelFactory
    ↓
Concrete Model
    ↓
Training
    ↓
Evaluator
    ↓
Prediction
```

### OOP Concepts Used

| Concept | Where Used |
|---|---|
| Encapsulation | `DataManager`, model state |
| Inheritance | Concrete classes from abstract bases |
| Polymorphism | Classification/Regression models |
| Abstraction | `Model`, `Preprocessor` |
| Magic methods | `__len__`, `__getitem__`, `__repr__`, `__call__`, `__bool__` |
| Composition | `MLApplication` |
| Dependency Injection | Constructor dependencies |
| Factory Pattern | `ModelFactory` |
| Custom Exceptions | Project-specific errors |
| SOLID | Separation of responsibilities |


## 16. Add a New Model

Try adding:

```text
DecisionTreeModel
NeuralNetworkModel
KNNModel
```

You should only need to implement the required model behavior.

The rest of the application should continue working without major changes.

This demonstrates the value of abstraction and the Open/Closed Principle.


In [ ]:
# YOUR TASK

class NewModel(Model):
    def train(self, X, y):
        # implement
        pass

    def predict(self, X):
        # implement
        pass

# Add the model to ModelFactory and test it through MLApplication.


## 17. Add a New Notification System

Extend the project with an abstract notification service:

```python
send(message)
```

Implement:

- `EmailNotification`
- `SMSNotification`
- `ConsoleNotification`

Inject the notification service into `MLApplication` and send a message after training.


In [ ]:
# YOUR TASK

class NotificationService(ABC):
    @abstractmethod
    def send(self, message):
        pass


class ConsoleNotification(NotificationService):
    def send(self, message):
        print("NOTIFICATION:", message)

# Extend MLApplication to accept and use a NotificationService.


## 18. Testing Ideas

Test these cases:

1. Empty dataset.
2. Mismatched X and y lengths.
3. Predicting before training.
4. Evaluating before training.
5. Unknown model type.
6. Adding a new model.
7. Replacing the preprocessor.
8. Replacing the evaluator.
9. Replacing the notification service.

Dependency injection makes these tests much easier.


## 19. Resume-Level Project Explanation

### Project Name
**OOP-Based Machine Learning Model Management System**

### Description
Designed an extensible Python OOP architecture for managing ML-style workflows using abstraction, inheritance, polymorphism, composition, dependency injection, custom exceptions, factory pattern, and Python magic methods.

### Key Features
- Pluggable model architecture
- Abstract preprocessing and model interfaces
- Model factory
- Dataset encapsulation
- Reusable evaluation service
- Dependency injection
- Custom exception handling
- Pythonic object behavior


## 20. Final Challenge

Upgrade this project into a more realistic ML application.

Add:

### Data Layer
- CSV loading
- train/test split

### Preprocessing
- missing-value handling
- scaling
- categorical encoding

### Models
- Logistic Regression
- Decision Tree
- Random Forest

### Evaluation
- accuracy
- precision
- recall
- F1-score

### Persistence
- save model
- load model

### Application
- prediction API/service
- logging
- configuration

Keep the architecture modular and use interfaces/abstractions where they provide real value.


# Final OOP Roadmap

```text
01 — OOP Basics
02 — Attributes & Methods
03 — Encapsulation
04 — Inheritance
05 — Polymorphism
06 — Abstraction
07 — Magic Methods
08 — Advanced OOP
09 — OOP Real-World Project
```

## Final Goal

You should now be able to move from:

**"I know OOP syntax"**

to:

**"I can design maintainable Python applications using OOP principles."**

The most important skill is not memorizing every feature. It is knowing **when to use inheritance, composition, abstraction, polymorphism, and dependency injection to solve a real problem cleanly.**
